## Quantum Fourier Transform & Algebraic Algorithms
These leverage the Quantum Fourier Transform (QFT) to convert state computational bases into [phase space](https://en.wikipedia.org/wiki/Phase_space), solving period-finding and factoring tasks with exponential speedup.

### 1. [Quantum Fourier Transform (QFT)](https://en.wikipedia.org/wiki/Quantum_Fourier_transform)
* **Problem Domain:** [Frequency Analysis](https://en.wikipedia.org/wiki/Frequency_analysis), Phase Manipulation.
* **Core Function:** Maps a computational state $\vert{}x\rangle$ to a Fourier basis state $\frac{1}{\sqrt{N}}\sum_{y=0}^{N-1} e^{2\pi i x y / N}\vert{}y\rangle$.
* **Algorithmic Mechanism:**
    1. Apply Hadamard gates sequentially to each qubit.
    2. Apply controlled phase rotations $CP(\theta_k)$ with decreasing angles $\theta_k = \frac{2\pi}{2^k}$.
    3. Swap qubit order at the end to align bit significance.

In [1]:
import numpy as np
from qiskit import QuantumCircuit

def create_qft(num_qubits):
    qc = QuantumCircuit(num_qubits, name="QFT")
    for i in range(num_qubits):
        qc.h(i)
        for j in range(i + 1, num_qubits):
            angle = np.pi / (2 ** (j - i))
            qc.cp(angle, j, i)
    # Swap qubits for correct bit ordering
    for i in range(num_qubits // 2):
        qc.swap(i, num_qubits - 1 - i)
    return qc

qft_3q = create_qft(3)
print("3-Qubit QFT Circuit:")
print(qft_3q.draw('text'))

3-Qubit QFT Circuit:
     ┌───┐                                        
q_0: ┤ H ├─■────────■───────────────────────────X─
     └───┘ │P(π/2)  │       ┌───┐               │ 
q_1: ──────■────────┼───────┤ H ├─■─────────────┼─
                    │P(π/4) └───┘ │P(π/2) ┌───┐ │ 
q_2: ───────────────■─────────────■───────┤ H ├─X─
                                          └───┘   


### 2. [Quantum Phase Estimation (QPE)](https://en.wikipedia.org/wiki/Quantum_phase_estimation_algorithm)
* **Problem Domain:** [Eigenvalue Determination](https://en.wikipedia.org/wiki/Eigenvalues_and_eigenvectors), [Quantum Chemistry](https://en.wikipedia.org/wiki/Quantum_chemistry).
* **Core Function:** Estimates phase $\theta$ of a unitary operator $U$ given eigenvector $\vert{}\psi\rangle$ such that $U\vert{}\psi\rangle = e^{2\pi i \theta}\vert{}\psi\rangle$.
* **Algorithmic Mechanism:**
    1. Initialize counting register in superposition and target register in state $\vert{}\psi\rangle$.
    2. Apply controlled unitary powers $C-U^{2^j}$ from counting qubits to the target register.
    3. Apply Inverse QFT ($QFT^\dagger$) to the counting register.
    4. Measure counting qubits to read phase $\theta$.

In [4]:
import numpy as np
from qiskit import QuantumCircuit

# QPE for T-gate (unitary phase theta = 1/8)
qc = QuantumCircuit(4, 3) # 3 counting qubits, 1 target qubit

# Initialize target in state |1>
qc.x(3)

# Step 1: Superposition on counting qubits
qc.h([0, 1, 2])

# Step 2: Controlled-U operations (T-gate powers)
qc.cp(np.pi / 4, 0, 3)          # C-U^(2^0)
qc.cp(2 * np.pi / 4, 1, 3)      # C-U^(2^1)
qc.cp(4 * np.pi / 4, 2, 3)      # C-U^(2^2)

# Step 3: Inverse QFT on counting qubits (simplified)
qc.h(0)
qc.cp(-np.pi / 2, 0, 1)
qc.h(1)
qc.cp(-np.pi / 4, 0, 2)
qc.cp(-np.pi / 2, 1, 2)
qc.h(2)

qc.measure([0, 1, 2], [0, 1, 2])

print("Quantum Phase Estimation Circuit:")
print(qc.draw('text'))

Quantum Phase Estimation Circuit:
     ┌───┐           ┌───┐                                     ┌─┐           
q_0: ┤ H ├─■─────────┤ H ├───■──────────────■──────────────────┤M├───────────
     ├───┤ │         └───┘   │P(-π/2) ┌───┐ │                  └╥┘     ┌─┐   
q_1: ┤ H ├─┼────────■────────■────────┤ H ├─┼─────────■─────────╫──────┤M├───
     ├───┤ │        │                 └───┘ │P(-π/4)  │P(-π/2)  ║ ┌───┐└╥┘┌─┐
q_2: ┤ H ├─┼────────┼─────────■─────────────■─────────■─────────╫─┤ H ├─╫─┤M├
     ├───┤ │P(π/4)  │P(π/2)   │P(π)                             ║ └───┘ ║ └╥┘
q_3: ┤ X ├─■────────■─────────■─────────────────────────────────╫───────╫──╫─
     └───┘                                                      ║       ║  ║ 
c: 3/═══════════════════════════════════════════════════════════╩═══════╩══╩═
                                                                0       1  2 


### 3. [Shor's Algorithm](https://en.wikipedia.org/wiki/Shor%27s_algorithm) (Period Finding Core)
* **Problem Domain:** Integer Factorization, Cryptanalysis (RSA/ECC).
* **Core Function:** Reduces integer factorization of $N$ to finding the order/period $r$ of $a^x \pmod N$ in polynomial time $\mathcal{O}(n^3)$.
* **Algorithmic Mechanism:**
    1. Choose a random integer $a < N$ coprime to $N$.
    2. Prepare an input register in equal superposition and a target register initialized to $\vert{}1\rangle$.
    3. Apply modular exponentiation $f(x) = a^x \pmod N$.
    4. Apply inverse QFT ($QFT^\dagger$) to the input register to measure phase $s/r$.
    5. Compute period $r$ using classical continued fractions and find factors $\gcd(a^{r/2} \pm 1, N)$.

In [1]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.circuit.library import QFT

# Simplified period finding setup for a^x mod N (a=2, N=3)
input_reg_size = 3
target_reg_size = 2
qc = QuantumCircuit(input_reg_size + target_reg_size, input_reg_size)

# Step 1: Superposition on input register & initialize target register to |1>
qc.h(range(input_reg_size))
qc.x(input_reg_size)

# Step 2: Modular Exponentiation Oracle U^{2^j} (Simplified modular multiplication)
qc.cx(0, input_reg_size)
qc.cx(1, input_reg_size + 1)

# Step 3: Inverse QFT on input register
iqft = QFT(num_qubits=input_reg_size, inverse=True).to_gate()
qc.append(iqft, range(input_reg_size))

# Step 4: Measure phase
qc.measure(range(input_reg_size), range(input_reg_size))

print("Shor's Period-Finding Subroutine:")
print(qc.draw('text'))

Shor's Period-Finding Subroutine:
     ┌───┐          ┌───────┐┌─┐      
q_0: ┤ H ├──■───────┤0      ├┤M├──────
     ├───┤  │       │       │└╥┘┌─┐   
q_1: ┤ H ├──┼────■──┤1 IQFT ├─╫─┤M├───
     ├───┤  │    │  │       │ ║ └╥┘┌─┐
q_2: ┤ H ├──┼────┼──┤2      ├─╫──╫─┤M├
     ├───┤┌─┴─┐  │  └───────┘ ║  ║ └╥┘
q_3: ┤ X ├┤ X ├──┼────────────╫──╫──╫─
     └───┘└───┘┌─┴─┐          ║  ║  ║ 
q_4: ──────────┤ X ├──────────╫──╫──╫─
               └───┘          ║  ║  ║ 
c: 3/═════════════════════════╩══╩══╩═
                              0  1  2 


C:\Users\tan\AppData\Local\Temp\ipykernel_22692\2989420481.py:19: DeprecationWarning: The class ``qiskit.circuit.library.basis_change.qft.QFT`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. ('Use qiskit.circuit.library.QFTGate or qiskit.synthesis.qft.synth_qft_full instead, for access to all previous arguments.',)
  iqft = QFT(num_qubits=input_reg_size, inverse=True).to_gate()


### 4. [Hadamard Test](https://en.wikipedia.org/wiki/Hadamard_test) & [Swap Test](https://en.wikipedia.org/wiki/Swap_test)
* **Problem Domain:** [Expectation Value](https://en.wikipedia.org/wiki/Expected_value) Estimation, [Quantum State Overlap/Similarity](https://en.wikipedia.org/wiki/Fidelity_of_quantum_states).
* **Core Function:** Calculates $\text{Re}\langle\psi\vert{}U\vert{}\psi\rangle$ (Hadamard Test) or state overlap $\vert{}\langle\psi\vert{}\phi\rangle\vert{}^2$ (Swap Test).
* **Algorithmic Mechanism (Swap Test):**
    1. Initialize an ancilla qubit in state $\vert{}0\rangle$ alongside state registers $\vert{}\psi\rangle$ and $\vert{}\phi\rangle$.
    2. Apply Hadamard gate to ancilla: $\frac{1}{\sqrt{2}}(\vert{}0\rangle + \vert{}1\rangle)\vert{}\psi\rangle\vert{}\phi\rangle$.
    3. Apply Controlled-SWAP ([Fredkin gate](https://en.wikipedia.org/wiki/Fredkin_gate)) conditioned on the ancilla between $\vert{}\psi\rangle$ and $\vert{}\phi\rangle$.
    4. Apply Hadamard to ancilla and measure in computational basis. Probability $P(\vert{}0\rangle) = \frac{1}{2} + \frac{1}{2}\vert{}\langle\psi\vert{}\phi\rangle\vert{}^2$.

In [2]:
from qiskit import QuantumCircuit

# Swap Test to measure similarity between |0> and |1>
qc = QuantumCircuit(3, 1)

# Prepare registers: Ancilla (0), Register A (1), Register B (2)
qc.x(2)  # Set Register B to state |1> (Register A remains |0>)

# Step 1: Ancilla Superposition
qc.h(0)

# Step 2: Controlled-SWAP (Fredkin Gate)
qc.cswap(0, 1, 2)

# Step 3: Hadamard on Ancilla & Measurement
qc.h(0)
qc.measure(0, 0)

print("Swap Test Circuit Layout:")
print(qc.draw('text'))

Swap Test Circuit Layout:
     ┌───┐   ┌───┐┌─┐
q_0: ┤ H ├─■─┤ H ├┤M├
     └───┘ │ └───┘└╥┘
q_1: ──────X───────╫─
     ┌───┐ │       ║ 
q_2: ┤ X ├─X───────╫─
     └───┘         ║ 
c: 1/══════════════╩═
                   0 


In [4]:
import numpy as np
from qiskit import QuantumCircuit

def hadamard_test(unitary_gate, part='real'):
    """
    Constructs a Hadamard Test circuit to estimate <ψ|U|ψ>.
    - part='real': Measures Re(<ψ|U|ψ>)
    - part='imag': Measures Im(<ψ|U|ψ>)
    """
    qc = QuantumCircuit(2, 1)  # Qubit 0: Ancilla, Qubit 1: State |ψ>
    
    # 1. Ancilla Superposition
    qc.h(0)
    
    # Phase shift for imaginary component calculation
    if part == 'imag':
        qc.sdg(0)
        
    # 2. Controlled Unitary execution C-U
    qc.append(unitary_gate.control(1), [0, 1])
    
    # 3. Measurement Phase
    qc.h(0)
    qc.measure(0, 0)
    
    return qc

# Example using a Z-gate as target Unitary U
from qiskit.circuit.library import ZGate

ht_real_circuit = hadamard_test(ZGate(), part='real')

print("Hadamard Test Circuit (Real Part):")
print(ht_real_circuit.draw('text'))

Hadamard Test Circuit (Real Part):
     ┌───┐   ┌───┐┌─┐
q_0: ┤ H ├─■─┤ H ├┤M├
     └───┘ │ └───┘└╥┘
q_1: ──────■───────╫─
                   ║ 
c: 1/══════════════╩═
                   0 


### 5. [Aharonov–Jones–Landau (AJL) Algorithm](https://en.wikipedia.org/wiki/Aharonov%E2%80%93Jones%E2%80%93Landau_algorithm)
* **Problem Domain:** [Topological Quantum Field Theory (TQFT)](https://en.wikipedia.org/wiki/Topological_quantum_field_theory), [Knot Theory](https://en.wikipedia.org/wiki/Knot_theory).
* **Core Function:** Polynomial-time approximation of topological invariants (such as the [Jones polynomial of knots](https://en.wikipedia.org/wiki/Jones_polynomial)) at roots of unity.
* **Algorithmic Mechanism:**
    1. Encodes braids into unitary representations of the braid group via [Temperley-Lieb algebras](https://en.wikipedia.org/wiki/Temperley%E2%80%93Lieb_algebra).
    2. Executes quantum circuit simulation of the braid matrix operations.
    3. Estimates the trace of the unitary operator using Hadamard test techniques to obtain polynomial values.

In [3]:
import numpy as np
from qiskit import QuantumCircuit

# Conceptual AJL braid unitary trace estimation via Hadamard Test
qc = QuantumCircuit(2, 1)

# Step 1: Ancilla Superposition
qc.h(0)

# Step 2: Braid Representation Unitary (Simulated via RZ/CX braid generator)
qc.crz(np.pi / 3, 0, 1)

# Step 3: Measure Ancilla for Trace Evaluation
qc.h(0)
qc.measure(0, 0)

print("AJL Knot Invariant (Trace Estimation) Unitary:")
print(qc.draw('text'))

AJL Knot Invariant (Trace Estimation) Unitary:
     ┌───┐           ┌───┐┌─┐
q_0: ┤ H ├─────■─────┤ H ├┤M├
     └───┘┌────┴────┐└───┘└╥┘
q_1: ─────┤ Rz(π/3) ├──────╫─
          └─────────┘      ║ 
c: 1/══════════════════════╩═
                           0 
